# 03 — Train GPU XGBoost and evaluate complete per-business matches

**Run All** retrieves a bounded training sample, extracts pair features, trains XGBoost on CUDA, tunes its decision threshold and evaluates a separate held-out group. Select **Python (ml-challenge-er)**.

02 achieved 95.88% candidate recall and an ideal-classifier ceiling of 0.9846 macro F0.5 on its 5,000-anchor pilot. It took about 62 minutes to retrieve that pilot. We reuse those candidates rather than repeating that work. **The default 3,000 new training anchors may take about 37 minutes of retrieval at that rate**, plus feature extraction and training. Progress/checkpoints allow resuming.

This is a first supervised baseline, not a final all-data training run. It creates a saved model and validation results; it does **not** generate test predictions or the two final submission TSVs.

All 5,000 pilot anchors stay out of model fitting. Half are used for early stopping/threshold tuning; the other half are used only for classifier evaluation. Retrieval was previously evaluated on the whole pilot, so this is an audit of the classifier, not an entirely untouched end-to-end holdout.

In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "8")
import sys, json, time, hashlib
from pathlib import Path
import numpy as np
import polars as pl
import xgboost as xgb
import rapidfuzz
from IPython.display import display

ancestors = (Path.cwd(), *Path.cwd().parents)
SRC = next((p / rel for p in ancestors for rel in
            (".", "src", "code/business_entity_resolution/src", "submission/code/business_entity_resolution/src")
            if (p / rel / "train_matcher.py").is_file()), None)
assert SRC is not None, "Open this notebook within the project or extracted submission package."
sys.path.insert(0, str(SRC.resolve()))
from train_matcher import (select_training_anchors, retrieve_training, load_labels, label_pairs,
                          feature_cache, entity_scores, tune_threshold, FEATURE_NAMES, PAIR_SCHEMA,
                          atomic_json, file_digest)
assert pl.__version__ == "1.44.2" and xgb.__version__ == "3.4.2"
assert np.__version__ == "2.4.6" and rapidfuzz.__version__ == "3.14.6"
print("Python:", sys.executable, "| XGBoost:", xgb.__version__)
# Check CUDA before spending time retrieving new candidates. Never silently fall back to CPU.
probe = xgb.XGBClassifier(n_estimators=2, max_depth=2, tree_method="hist", device="cuda", n_jobs=8)
probe.fit(np.array([[0,0],[1,1],[0,1],[1,0]], dtype=np.float32), np.array([0,1,0,1]))
actual_device = json.loads(probe.get_booster().save_config())["learner"]["generic_param"]["device"]
assert actual_device.startswith("cuda"), f"XGBoost used {actual_device}. Fix GPU access before continuing."
print("Verified GPU:", actual_device)
del probe


Python: /home/aline/.conda/envs/ml-challenge-er/bin/python | XGBoost: 3.4.2
Verified GPU: cuda:0


In [2]:
CACHE_ROOT = (Path(os.environ["ER_CACHE_ROOT"]).expanduser().resolve() if os.environ.get("ER_CACHE_ROOT") else
              next((p / "cache/normalized_v1" for p in ancestors
                    if (p / "cache/normalized_v1/train/train_source1/manifest.json").is_file()), None))
assert CACHE_ROOT is not None, "Set ER_CACHE_ROOT to notebook 01's completed cache."
CANDIDATE_ROOT = Path(os.environ.get("ER_CANDIDATE_ROOT", str(CACHE_ROOT.parent / "candidates_v1"))).expanduser().resolve()
TRAIN_ROOT = Path(os.environ.get("ER_TRAIN_ROOT", str(CACHE_ROOT.parent / "training_v1"))).expanduser().resolve()
TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIR = CANDIDATE_ROOT / "pilot"
INDEX = CANDIDATE_ROOT / "train_search.sqlite"
N_TRAIN_ANCHORS = 3_000  # First baseline. Changing this requires a new ER_TRAIN_ROOT.
RETRIEVAL_BATCH_ANCHORS = 100
MODEL_DIR = TRAIN_ROOT / "model"
MODEL_DIR.mkdir(exist_ok=True)
pilot_metrics = json.loads((PILOT_DIR / "pilot_metrics.json").read_text())
pilot_details = pl.read_parquet(PILOT_DIR / "pilot_recall.parquet")
pilot_pairs = pl.read_parquet(PILOT_DIR / "pilot_candidates.parquet")
assert pilot_pairs.height == pilot_metrics["candidate_pairs"]
assert pilot_details.height == pilot_metrics["anchors"]
assert pilot_details["source1_entity_id"].n_unique() == pilot_details.height
assert pilot_details.select((pl.col("source1_entity_id").hash(seed=42)%10 == 0).all()).item()
RETRIEVAL = pilot_metrics["retriever_config"]
print("Training work directory:", TRAIN_ROOT)
print(f"Existing candidate recall: {pilot_metrics['pair_recall']:.4%}")
print(f"Retrieval-only F0.5 ceiling on the 02 pilot: {pilot_metrics['oracle_macro_f05_ceiling']:.6f}")
print(f"New training retrieval estimate: {pilot_metrics['retrieval_seconds']/pilot_metrics['anchors']*N_TRAIN_ANCHORS/60:.1f} minutes at the 02 rate.")


Training work directory: /home/aline/Projects/ML-Challenge/cache/training_v1
Existing candidate recall: 95.8833%
Retrieval-only F0.5 ceiling on the 02 pilot: 0.984581
New training retrieval estimate: 37.0 minutes at the 02 rate.


## Split businesses before training

Training anchors come from the 90% partition excluded from 02's validation sample. We sample 3,000 deterministically. Pilot businesses are split by another deterministic hash into **tune** and **audit** groups. Entire businesses stay together; we never randomly split individual candidate pairs.

The tune set controls early stopping and the decision threshold. The audit set is scored only after both are fixed. No country one-hot encoding or raw entity ID is a model feature. France remains unseen here.

In [3]:
train_anchors = select_training_anchors(CACHE_ROOT, N_TRAIN_ANCHORS)
train_roles = train_anchors.select(pl.col("entity_id").alias("source1_entity_id")).with_columns(pl.lit("train").alias("role"))
validation_roles = pilot_details.select("source1_entity_id").with_columns(
    pl.when(pl.col("source1_entity_id").hash(seed=77)%2 == 0).then(pl.lit("tune"))
      .otherwise(pl.lit("audit")).alias("role"))
roles = pl.concat([train_roles, validation_roles])
assert roles["source1_entity_id"].n_unique() == roles.height, "Business overlap between train and validation."
assert train_anchors.select((pl.col("entity_id").hash(seed=42)%10 != 0).all()).item()
assert set(roles["role"]) == {"train", "tune", "audit"}
roles.write_parquet(TRAIN_ROOT / "anchor_roles.parquet")
display(roles.group_by("role").len().sort("role"))


role,len
str,u32
"""audit""",2458
"""train""",3000
"""tune""",2542


## Retrieve realistic training candidates

This is the slow cell. It applies **the same retrieval settings as 02** to new training anchors, and commits every 100 anchors. Interrupt and rerun to resume. Avoid concurrent writers to the same directory.

True matches are not injected. Negative examples are plausible retrieved businesses that the labels say are different, rather than unrelated random rows. Missing positive candidates are still counted as misses during evaluation.

In [4]:
train_pairs = retrieve_training(train_anchors, INDEX, RETRIEVAL,
                                TRAIN_ROOT / "train_candidates", RETRIEVAL_BATCH_ANCHORS)
pairs = pl.concat([train_pairs.select(list(PAIR_SCHEMA)), pilot_pairs.select(list(PAIR_SCHEMA))])
assert pairs.select(pl.struct("source1_entity_id", "candidate_entity_id").n_unique()).item() == pairs.height
assert pairs["source1_entity_id"].is_in(roles["source1_entity_id"].to_list()).all()
print(f"Training pairs: {train_pairs.height:,}; pilot pairs reused: {pilot_pairs.height:,}")


Training retrieval: 100/3,000 anchors | ETA 36.6 min
Training retrieval: 200/3,000 anchors | ETA 35.7 min
Training retrieval: 300/3,000 anchors | ETA 33.9 min
Training retrieval: 400/3,000 anchors | ETA 32.3 min
Training retrieval: 500/3,000 anchors | ETA 31.0 min
Training retrieval: 600/3,000 anchors | ETA 29.8 min
Training retrieval: 700/3,000 anchors | ETA 28.5 min
Training retrieval: 800/3,000 anchors | ETA 27.2 min
Training retrieval: 900/3,000 anchors | ETA 25.9 min
Training retrieval: 1,000/3,000 anchors | ETA 24.7 min
Training retrieval: 1,100/3,000 anchors | ETA 23.4 min
Training retrieval: 1,200/3,000 anchors | ETA 22.1 min
Training retrieval: 1,300/3,000 anchors | ETA 20.9 min
Training retrieval: 1,400/3,000 anchors | ETA 19.7 min
Training retrieval: 1,500/3,000 anchors | ETA 18.4 min
Training retrieval: 1,600/3,000 anchors | ETA 17.2 min
Training retrieval: 1,700/3,000 anchors | ETA 16.0 min
Training retrieval: 1,800/3,000 anchors | ETA 14.7 min
Training retrieval: 1,900/3,

## Labels and 43 pair-comparison features

Features compare full/core/original-script names and addresses, token overlap, number agreement/conflict, missingness, script flags and retrieval evidence. A number conflict is evidence, not a hard rejection. Empty fields never produce a perfect similarity feature.

All features are computed without labels; the labels are joined afterward. Reference and target IDs are metadata, not features. Derived feature shards are cached for reruns. All features are float32 to bound memory.

In [5]:
labels = load_labels(CACHE_ROOT, roles["source1_entity_id"].to_list())
labeled = label_pairs(pairs, labels)
features = feature_cache(CACHE_ROOT, pairs, TRAIN_ROOT / "features")
data = (features.join(labeled.select("source1_entity_id", "candidate_entity_id", "label"),
                      on=["source1_entity_id", "candidate_entity_id"], how="left")
        .join(roles, on="source1_entity_id", how="left"))
assert data.height == pairs.height and data["label"].null_count() == 0 and data["role"].null_count() == 0
assert not {"label", "source1_entity_id", "candidate_entity_id", "role"}.intersection(FEATURE_NAMES)
display(data.group_by("role").agg(pl.len().alias("pairs"), pl.col("label").sum().alias("positive_pairs")))
print("Feature count:", len(FEATURE_NAMES))


Features: 25,000/385,299 pairs
Features: 50,000/385,299 pairs
Features: 75,000/385,299 pairs
Features: 100,000/385,299 pairs
Features: 125,000/385,299 pairs
Features: 150,000/385,299 pairs
Features: 175,000/385,299 pairs
Features: 200,000/385,299 pairs
Features: 225,000/385,299 pairs
Features: 250,000/385,299 pairs
Features: 275,000/385,299 pairs
Features: 300,000/385,299 pairs
Features: 325,000/385,299 pairs
Features: 350,000/385,299 pairs
Features: 375,000/385,299 pairs
Features: 385,299/385,299 pairs


role,pairs,positive_pairs
str,u32,i64
"""train""",144336,9937
"""tune""",122826,8434
"""audit""",118137,8266


Feature count: 43


In [6]:
train_data = data.filter(pl.col("role") == "train")
tune_data = data.filter(pl.col("role") == "tune")
audit_data = data.filter(pl.col("role") == "audit")
assert train_data["label"].n_unique() == 2 and tune_data["label"].n_unique() == 2
X_train = train_data.select(FEATURE_NAMES).to_numpy().astype(np.float32, copy=False)
X_tune = tune_data.select(FEATURE_NAMES).to_numpy().astype(np.float32, copy=False)
y_train = train_data["label"].to_numpy()
y_tune = tune_data["label"].to_numpy()
# Give each represented training business equal total weight, rather than favoring larger candidate sets.
counts = train_data.group_by("source1_entity_id").len().rename({"len": "candidate_count"})
weights = train_data.select("source1_entity_id").join(counts, on="source1_entity_id", maintain_order="left")["candidate_count"].to_numpy()
weights = (1.0 / weights).astype(np.float32)
weights /= weights.mean()
PARAMS = dict(objective="binary:logistic", tree_method="hist", device="cuda", n_jobs=8,
              n_estimators=1200, learning_rate=0.05, max_depth=6, min_child_weight=5,
              subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0,
              eval_metric="logloss", early_stopping_rounds=60, random_state=42)
model = xgb.XGBClassifier(**PARAMS)
started = time.perf_counter()
model.fit(X_train, y_train, sample_weight=weights, eval_set=[(X_tune, y_tune)], verbose=100)
actual_device = json.loads(model.get_booster().save_config())["learner"]["generic_param"]["device"]
assert actual_device.startswith("cuda"), "Training did not use CUDA."
print(f"Fit finished in {time.perf_counter()-started:.1f}s; best iteration: {model.best_iteration}")
model.save_model(MODEL_DIR / "xgboost_model.json")


[0]	validation_0-logloss:0.21449
[100]	validation_0-logloss:0.03121
[200]	validation_0-logloss:0.02733
[300]	validation_0-logloss:0.02627
[400]	validation_0-logloss:0.02587
[500]	validation_0-logloss:0.02567
[600]	validation_0-logloss:0.02563
[617]	validation_0-logloss:0.02564
Fit finished in 1.2s; best iteration: 557


## Tune threshold using the exact macro F0.5 objective

For each business, score = `1.25 × TP / (0.25 × true_count + predicted_count)`.
If both counts are zero, score = 1. Missing true candidates remain in `true_count`.
All businesses are included, even when retrieval found no candidates.

We sweep thresholds on **tune only**. The threshold above 1 represents predicting no matches for every business. Ties prefer the higher threshold. XGBoost early stopping uses tune log loss; final acceptance uses per-business macro F0.5.

In [7]:
tune_prob = model.predict_proba(X_tune)[:, 1]
tune_labels = labels.join(roles.filter(pl.col("role") == "tune"), on="source1_entity_id", how="semi")
pair_keys = ["source1_entity_id", "candidate_entity_id", "label"]
threshold, threshold_table = tune_threshold(tune_data.select(pair_keys), tune_labels, tune_prob)
threshold_table.write_csv(MODEL_DIR / "threshold_sweep.tsv", separator="\t")
print("Selected threshold:", threshold)
display(threshold_table.head(10))


/home/aline/.conda/envs/ml-challenge-er/lib/python3.11/site-packages/xgboost/core.py:569: UserWarning: [23:02:44] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1789510313506/work/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Selected threshold: 0.64


threshold,macro_f05
f64,f64
0.64,0.918506
0.63,0.918417
0.58,0.918343
0.62,0.918335
0.57,0.918014
0.59,0.918003
0.61,0.917954
0.6,0.917898
0.56,0.917474


In [8]:
# Do not change the threshold using this audit result.
audit_labels = labels.join(roles.filter(pl.col("role") == "audit"), on="source1_entity_id", how="semi")
audit_prob = model.predict_proba(audit_data.select(FEATURE_NAMES).to_numpy().astype(np.float32, copy=False))[:, 1]
audit_scores = entity_scores(audit_data.select(pair_keys), audit_labels, audit_prob, threshold)
audit_scores = audit_scores.join(pilot_details.select("source1_entity_id", "country", "oracle_macro_f05"), on="source1_entity_id")
audit_scores.write_parquet(MODEL_DIR / "audit_entity_scores.parquet")
audit_data.select(pair_keys).with_columns(pl.Series("probability", audit_prob)).write_parquet(MODEL_DIR / "audit_pair_scores.parquet")
report = {"training_anchors": train_roles.height, "training_pairs": train_data.height,
          "tune_anchors": tune_labels.height, "audit_anchors": audit_labels.height,
          "threshold": threshold, "tune_macro_f05": threshold_table["macro_f05"][0],
          "audit_macro_f05": audit_scores["f05"].mean(),
          "audit_retrieval_oracle_ceiling": audit_scores["oracle_macro_f05"].mean(),
          "audit_all_empty_baseline": (audit_scores["true_count"] == 0).mean(),
          "best_iteration": int(model.best_iteration), "device": actual_device,
          "note": "Classifier audit on part of the previously inspected 02 pilot; not leaderboard/test performance."}
atomic_json(MODEL_DIR / "validation_metrics.json", report)
atomic_json(MODEL_DIR / "model_metadata.json", {
    "feature_names": FEATURE_NAMES, "threshold": threshold, "parameters": PARAMS,
    "retrieval_config": RETRIEVAL, "versions": {"xgboost": xgb.__version__, "polars": pl.__version__,
                                                "numpy": np.__version__, "rapidfuzz": rapidfuzz.__version__},
    "feature_code_sha256": file_digest(SRC / "train_matcher.py"),
    "retrieval_code_sha256": file_digest(SRC / "candidate_retrieval.py"),
    "anchor_roles_sha256": file_digest(TRAIN_ROOT / "anchor_roles.parquet"),
    "pilot_candidates_sha256": file_digest(PILOT_DIR / "pilot_candidates.parquet"),
    "best_iteration": int(model.best_iteration), "validation": report})
print(json.dumps(report, indent=2))
display(audit_scores.group_by("country").agg(pl.len().alias("anchors"), pl.col("f05").mean().alias("macro_f05"),
                                            pl.col("oracle_macro_f05").mean().alias("retrieval_ceiling")))
display(pl.DataFrame({"feature": FEATURE_NAMES, "importance": model.feature_importances_}).sort("importance", descending=True).head(15))
print("Saved model and metrics in:", MODEL_DIR)


{
  "training_anchors": 3000,
  "training_pairs": 144336,
  "tune_anchors": 2542,
  "audit_anchors": 2458,
  "threshold": 0.64,
  "tune_macro_f05": 0.9185057501784811,
  "audit_macro_f05": 0.9237091328018934,
  "audit_retrieval_oracle_ceiling": 0.9860353656066716,
  "audit_all_empty_baseline": 0.05532953620829943,
  "best_iteration": 557,
  "device": "cuda:0",
  "note": "Classifier audit on part of the previously inspected 02 pilot; not leaderboard/test performance."
}


country,anchors,macro_f05,retrieval_ceiling
str,u32,f64,f64
"""us""",1488,0.93844,0.994905
"""india""",970,0.901111,0.972429


feature,importance
str,f32
"""log_bm25_magnitude""",0.176346
"""address_token_set""",0.115832
"""reciprocal_rank""",0.09833
"""address_containment""",0.089549
"""digit_jaccard""",0.074933
…,…
"""address_missing_b""",0.024428
"""route_joint""",0.021213
"""name_token_sort""",0.020271


Saved model and metrics in: /home/aline/Projects/ML-Challenge/cache/training_v1/model


## Next step

Share `model/validation_metrics.json` and the country table. Compare classifier performance with the retrieval ceiling. Retrieval improvements and/or more training data may be needed; a trained model is not proof of leaderboard quality.

For later inference, load the saved model **with the saved feature order and threshold**, use the same normalization and retrieval semantics, and honor the saved best iteration. The packaged reproduction code is in `src/`; generated models and caches currently remain outside the submission folder. Final packaging must add any required model artifacts or complete retraining instructions.

The two final TSVs are still outstanding. In particular, 03 does not run millions of test searches at the current slow retrieval rate. We should address that cost before scheduling full test inference.